# Demo — A Data Contract that Carries Policy

A data contract turns governance intent into something a pipeline can **validate**. This
walkthrough shows the technique on the **`shipment`** dataset (`Inventory & Fulfillment` domain):
start from an ungoverned contract, watch it fail validation, see how `validate_contract()`
enforces both **JSON Schema structure** and **semantic cross-checks**, and construct a compliant
contract so it passes. The exercise asks you to govern the Trailhead `orders` contract and
explain how a policy change propagates across downstream consumers.

In [1]:
from governance_toolkit import validate_contract, CONTRACT_SCHEMA

shipment_contract = {
    "name": "shipment",
    "domain": "Inventory & Fulfillment",
    "owner": "inventory-fulfillment@trailhead.example",
    "version": "1.0.0",
    "columns": [
        {"name": "shipment_id", "type": "integer"},
        {"name": "order_id", "type": "integer"},
        {"name": "carrier", "type": "string"},
        {"name": "ship_ts", "type": "timestamp"},
        {"name": "carbon_kg", "type": "double"},
    ],
}
ok, errs = validate_contract(shipment_contract)
print("Valid?", ok, "| errors:", errs)   # fails: no governance block yet

Valid? False | errors: ["(root): 'governance' is a required property"]


## What the validator enforces (schema structure + semantic rules)
`validate_contract()` checks two layers:
1. **JSON Schema (`CONTRACT_SCHEMA`):** every contract must declare `governance` with `pii_tags`,
   an ISO-8601 `retention` duration (`^P\d+[YMD]$`), and at least one `quality_slos` entry with a
   numeric `threshold` in `(0, 1]`.
2. **Semantic cross-checks:** every entry in `governance["pii_tags"]` must exist in `columns` and
   match the columns marked with `col["pii"] = True`.

Watch what happens when we pass an informal retention string (`"10 years"`), a percentage threshold
(`100` instead of `1.0`), and a `pii_tags` list without marking the column in `columns`:

In [2]:
shipment_contract["governance"] = {
    "pii_tags": ["order_id"],
    "retention": "10 years",  # invalid: must be ISO-8601 duration (e.g. P10Y)
    "quality_slos": [
        {"dimension": "accuracy", "threshold": 100},  # invalid: threshold must be in (0, 1]
    ],
}
ok, errs = validate_contract(shipment_contract)
print("Valid with bad fields?", ok)
for e in errs:
    print(" -", e)

Valid with bad fields? False
 - governance/quality_slos/0/threshold: 100 is greater than the maximum of 1.0
 - governance/retention: '10 years' does not match '^P\\d+[YMD]$'
 - columns: at least one PII column must be marked with col['pii'] = True
 - governance/quality_slos/0/threshold: 100.0 must be between 0 and 1 (exclusive of 0)


## Construct a compliant `shipment` contract
Fix the ISO-8601 retention format (`"P10Y"` for 10-year CSRD sustainability audit retention), set
the `accuracy` SLO threshold on `carbon_kg` to `1.0` (`100%` in kilograms), and align
`col["pii"] = True` on `order_id` (an indirect link to a customer's purchase) with `pii_tags`:

In [3]:
shipment_contract["columns"][1]["pii"] = True   # order_id links shipment to customer order

shipment_contract["governance"] = {
    "pii_tags": ["order_id"],
    "retention": "P10Y",
    "quality_slos": [
        {"dimension": "accuracy", "threshold": 1.0},
    ],
}
ok, errs = validate_contract(shipment_contract)
print("Valid now?", ok, "| errors:", errs)
shipment_contract["governance"]

Valid now? True | errors: []


{'pii_tags': ['order_id'], 'retention': 'P10Y', 'quality_slos': [{'dimension': 'accuracy', 'threshold': 1.0}]}

**Takeaway / your turn:** a governed contract makes policy machine-checkable — a CI
gate can refuse to publish a product whose contract fails schema or semantic validation, and
downstream systems inherit the declared `retention` and `pii_tags` obligations. In the exercise
you'll govern the `orders` contract yourself and explain how its PII and retention policies
propagate to producers and downstream consumers.